# 19. More seeds of the target-encoded CatBoost

Seed 42 is ledger row 26. This runs seeds 2024, 7, 2025 and 13 with everything else
identical, giving a CatBoost seed set that matches the LightGBM one from `15`
exactly: same seeds, same folds, same encoder, same budget.

## The question it answers

Row 26 recorded CatBoost at CV 0.966915 against row 17's 0.966782, **+0.000132
paired, 5/5 folds**, and logged it as **parity rather than an improvement** because it
fails two of the three tests in `NOTES.md`. One of those two is fixable and this run
fixes it: **only one CatBoost seed had been run**, so the difference could not be
separated from CatBoost's own seed variation, which nothing in this repo has measured.

The LightGBM target-encoded seeds span **6e-05** across five seeds, 0.966729 to
0.966789. If CatBoost's spread is comparable, +0.000132 sits well outside it and row
26's parity claim resolves upward. If CatBoost's spread is several times wider, the
claim stays where it is. **Both outcomes are useful and the ledger takes whichever
comes out.**

The second failed test, that the gain is smaller than the fold spread, is not
fixable and is not meant to be: `NOTES.md` already records that fold spread is the
wrong yardstick for a paired comparison, because it is common to both models and
cancels in the difference.

## What is held fixed

The encoder's inner split is deliberately **not** reseeded. Only CatBoost's
`random_seed` varies, so this is a seed sweep of the model and not of the
representation as well. Same choice `15` made, for the same reason.

Four seeds at about 65 minutes each. The four new out-of-fold vectors also become
stack members, which is what the equivalent LightGBM seeds did in rows 20 to 23.


In [ ]:
# Set False for the real run. Smoke mode exercises every line on 20k rows.
SMOKE = True

# Seed 42 is ledger row 26 and is not re-run. Same list as 15.
SEEDS = [2024, 7, 2025, 13]

# The fold split and the encoder's inner split. Held fixed across every model seed,
# so this sweep varies the model's stochasticity and not the representation.
# It has to be called SEED rather than something more descriptive: the encoder is
# copied from 13 and its default argument reads `seed=SEED`, so renaming it changes
# the semantic fingerprint and the check below fails. That check caught exactly this
# on the first smoke run.
SEED = 42
N_INNER = 5
SMOOTH = 10.0
THREADS = 6
LR = 0.05
ITERS = 2000

EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# Ledger row 26, CatBoost seed 42, and row 17, the LightGBM it is compared against.
ROW26_CV = 0.966915
ROW17_CV = 0.966782
# The five LightGBM target-encoded seeds from rows 17 and 20 to 23, for the spread
# comparison that is the point of this run.
LGB_SEED_CV = {42: 0.966782, 2024: 0.966771, 7: 0.966729,
               2025: 0.966743, 13: 0.966789}

print(f"SMOKE = {SMOKE}   seeds to run: {SEEDS}")


In [ ]:
import ast
import hashlib
import time
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    ITERS = 100
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


## The encoder, and the check that it is the same one

Copied from `13_target_encoding.ipynb`, like `17`. The copy is fingerprinted through
`ast.unparse`, which strips comments and formatting but keeps semantics, so a drifted
copy is caught rather than assumed away. Under the notebook layout this is the
substitute for a config hash.


In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17 and 26's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - these seeds are NOT comparable")


In [ ]:
# The same three leak checks 13 ran, on the same encoder, so their numbers are
# directly comparable to the ones in NOTES.md. Read together: the first two must be
# about zero, the third must be large.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()


In [ ]:
LOG = OUT / "19_catboost_seeds.log"


def mem():
    try:
        if not hasattr(__import__("os"), "add_dll_directory"):
            for line in Path("/proc/meminfo").read_text().splitlines():
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 2 ** 20
            return None
        import ctypes

        class MS(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong),
                        ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong),
                        ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong),
                        ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong),
                        ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]

        m = MS()
        m.dwLength = ctypes.sizeof(MS)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
        return m.ullAvailPhys / 2 ** 30
    except Exception:
        return None


def note(msg):
    """Print, and append to a log flushed on every write.

    nbconvert and the Kaggle runner both write the notebook only once the whole run
    finishes, so without this a four hour run is opaque from outside the kernel.
    """
    print(msg)
    g = mem()
    tail = "" if g is None else f"  [{g:.1f} GB free]"
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}{tail}", file=fh, flush=True)


def to_cb(df):
    """CatBoost wants categoricals as strings with no NaN."""
    d = df.copy()
    for c in CAT:
        d[c] = d[c].astype("object").fillna("__NA__").astype(str)
    return d


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


def run_seed(model_seed):
    """Five folds at one CatBoost seed. The encoder is not reseeded."""
    oof = np.zeros(len(train))
    tst = np.zeros(len(test))
    sc = []
    for f in range(5):
        tr = np.where(folds != f)[0]
        va = np.where(folds == f)[0]
        Xtr, Xva, Xte = build(X, y, tr, va, X_test)
        Xtr, Xva, Xte = to_cb(Xtr), to_cb(Xva), to_cb(Xte)
        cat_idx = [Xtr.columns.get_loc(c) for c in CAT]
        m = cb.CatBoostClassifier(
            iterations=ITERS, learning_rate=LR, random_seed=model_seed,
            thread_count=THREADS, allow_writing_files=False, verbose=0,
        )
        m.fit(Xtr, y[tr], cat_features=cat_idx)
        oof[va] = m.predict_proba(Xva)[:, 1]
        tst += m.predict_proba(Xte)[:, 1] / 5
        sc.append(roc_auc_score(y[va], oof[va]))
    return float(np.mean(sc)), float(np.std(sc)), oof, tst


note(f"=== run start, SMOKE={SMOKE}, seeds={SEEDS}, threads={THREADS} ===")


In [ ]:
results = {}
t0 = time.time()
for s in SEEDS:
    ts = time.time()
    cv, sd, oof, tst = run_seed(s)
    results[s] = (cv, sd, oof, tst)
    note(f"seed {s:5d}: CV {cv:.6f} +/- {sd:.6f}  ({hhmm(time.time() - ts)}), "
         f"elapsed {hhmm(time.time() - t0)}")
print()
print(f"{len(results)} seeds in {hhmm(time.time() - t0)}")


In [ ]:
# The comparison this run exists for. Seed 42 comes from row 26's saved vector rather
# than being retrained, and it is checked against its ledger number first, because a
# spread computed against a vector that does not reproduce would be meaningless.
cat_cv = {}
try:
    o42 = np.load(locate("catboost_te_oof.npy"))[ROW_IDX]
    cv42 = float(np.mean([roc_auc_score(y[folds == f], o42[folds == f])
                          for f in range(5)]))
    print(f"row 26 seed 42 vector reproduces its ledger CV to "
          f"{cv42 - ROW26_CV:+.2e}" if not SMOKE else
          f"SMOKE: seed 42 vector on the subsample scores {cv42:.6f}, not comparable")
    if not SMOKE:
        cat_cv[42] = cv42
except FileNotFoundError:
    print("row 26's out-of-fold vector not found, so seed 42 is quoted from the ledger")
    if not SMOKE:
        cat_cv[42] = ROW26_CV

for s, (cv, _, _, _) in results.items():
    cat_cv[s] = cv

cat = np.array([cat_cv[s] for s in sorted(cat_cv)])
lgb = np.array(sorted(LGB_SEED_CV.values()))
print()
print(f"{'seed':>6}  {'CatBoost':>10}  {'LightGBM':>10}")
for s in sorted(cat_cv):
    lg = LGB_SEED_CV.get(s)
    print(f"{s:>6}  {cat_cv[s]:>10.6f}  " + (f"{lg:>10.6f}" if lg else " " * 10))
print()
print(f"CatBoost seeds: mean {cat.mean():.6f}, range {cat.max() - cat.min():.2e}, "
      f"sd {cat.std(ddof=1):.2e}  (n={len(cat)})")
print(f"LightGBM seeds: mean {lgb.mean():.6f}, range {lgb.max() - lgb.min():.2e}, "
      f"sd {lgb.std(ddof=1):.2e}  (n={len(lgb)})")
print()
gap = cat.mean() - lgb.mean()
pooled = np.sqrt(cat.var(ddof=1) / len(cat) + lgb.var(ddof=1) / len(lgb))
print(f"gap between the family means: {gap:+.6f}")
print(f"standard error of that gap  : {pooled:.6f}   ->  {gap / pooled:.1f} se")
print()
print("This is the number row 26 could not compute. It does not replace row 26's")
print("paired per-fold comparison, which is tighter; it answers the separate question")
print("of whether one CatBoost seed could have produced +0.000132 by chance.")


In [ ]:
prefix = "SMOKE_" if SMOKE else ""
for s, (cv, sd, oof, tst) in results.items():
    np.save(OUT / f"{prefix}catboost_te_seed{s}_oof.npy", oof)
    np.save(OUT / f"{prefix}catboost_te_seed{s}_test.npy", tst)
print(f"saved {len(results)} seed vectors to {OUT}")
print()
print(f"leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}")
print()
for s, (cv, sd, _, _) in results.items():
    print(f"  ledger line: catboost_te_seed{s}   cv {cv:.6f} +/- {sd:.6f}")
print()
print("Stacking these is done locally in a follow-up notebook, where every other")
print("member vector lives.")
